In [ ]:
#if (!require("BiocManager", quietly = TRUE))
    #install.packages("BiocManager")

#BiocManager::install("MPRAnalyze")
#Format the data into MPRA project
library("MPRAnalyze")
library("BiocParallel")

dropEnhancer <- function(df_RNA){
    row.names(df_RNA) <-df_RNA$enhancer_id	
    df <- df_RNA[ , !(names(df_RNA) %in% c("enhancer_id"))]
    return(df)
    }
    
dropX <- function(df_RNA){
    row.names(df_RNA) <-df_RNA$X
    df <- df_RNA[ , !(names(df_RNA) %in% c("X"))]
    return(df)
    }
    
# Function to load data and convert it to matrix after dropping enhancer
load_and_process <- function(filepath, drop_func) {
  df <- read.csv(filepath, header = TRUE)
  as.matrix(drop_func(df))
}
register(MulticoreParam(24))
bpparam <- MulticoreParam(24, log = TRUE, stop.on.error = FALSE)


In [ ]:
process_datasets <- function(dna_file, rna_file, annot_file, neg_control_file) {
  # Function to drop enhancer or any specific column and set row names
  drop_and_rename <- function(df, col_name) {
    if (col_name %in% names(df)) {
      if (any(is.na(df[[col_name]])) || any(duplicated(df[[col_name]]))) {
        stop(paste("Column", col_name, "contains NA or duplicate values."))
      }
      row.names(df) <- df[[col_name]]
      df <- df[, !(names(df) %in% c(col_name))]
    }
    df
  }

  # Load and process CSV files
  load_and_process <- function(filepath, drop_func, col_name) {
    df <- read.csv(filepath, header = TRUE)
    drop_func(df, col_name)
  }

  # Load data
  df_DNA <- load_and_process(dna_file, drop_and_rename, "enhancer_id")
  df_RNA <- load_and_process(rna_file, drop_and_rename, "enhancer_id")
  annot_DNA <- load_and_process(annot_file, drop_and_rename, "X")

  # Calculate selected columns for subsetting
  total_columns <- ncol(df_DNA)
  selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]

  # Subset data starting from the 32nd row
  df_DNA <- df_DNA[, selected_columns]
  df_RNA <- df_RNA[, selected_columns]
  annot_DNA <- annot_DNA[selected_columns,]

  # Convert columns to factor for MPRAnalyze
  annot_DNA[] <- lapply(annot_DNA, as.factor)

  # Load negative controls and extract sequence IDs
  negative <- read.csv(neg_control_file, sep = "\t", header = FALSE)
  control <- as.character(negative$V1)  # V1 is the sequence_ID
  
  # Return a list of processed data frames and control
  list(DNA = as.matrix(df_DNA), RNA = as.matrix(df_RNA), Annotation = annot_DNA, Control = control)
}

In [ ]:
df_DNA <- read.csv("read_counts_R1R2/allele_only/THP1Macrophage_DNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/allele_only/THP1Macrophage_RNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_THP1Macrophage_barcodes.csv", header=TRUE,row.names=1)

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID

##############################################################
total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]
df_DNA <- df_DNA[, selected_columns]
df_RNA <- df_RNA[, selected_columns]
annot_DNA <- annot_DNA[selected_columns,]
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )

obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String,
                          reducedDesign = ~ Tissue,
                          )                     
res <-testLrt(obj)
write.csv(res,"20240813_comparative_THP1Macrophage_alleleOnly.csv")


Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-08-13 11:39:58.11587
Success: TRUE

Task duration:
   user  system elapsed 
259.173   2.033 290.381 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6175702 329.9   11606934 619.9  8605685 459.6
Vcells 13757620 105.0   27813248 212.2 27802250 212.2

Log messages:
INFO [2024-08-13 11:35:07] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 15
Node: 10
Timestamp: 2024-08-13 11:41:11.643937
Success: TRUE

Task duration:
   user  system elapsed 
335.611   1.858 364.229 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6176981 329.9   11606934 619.9  8605685 459.6
Vcells 13764439 105.1   27813248 212.2 27802250 212.2

Log messages:
INFO [2024-08-13 11:35:07] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 14
Node: 11
Timestamp: 2024-08-13 11:

In [ ]:


results <- process_datasets(
    "read_counts_R1R2/allele_only/THP1_Naive_DNA_matched_barcodes_reshaped_allele.csv",
    "read_counts_R1R2/allele_only/THP1_Naive_RNA_matched_barcodes_reshaped_allele.csv",
    "annotation_barcodes/mpra3_annot_THP1_Naive_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )

obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          )                         
res <-testLrt(obj)
write.csv(res,"20240616_comparative_THP1_Naive_alleleOnly.csv")

results <- process_datasets(
    "read_counts_R1R2/allele_only/THP1_IFNB_DNA_matched_barcodes_reshaped_allele.csv",
    "read_counts_R1R2/allele_only/THP1_IFNB_RNA_matched_barcodes_reshaped_allele.csv",
    "annotation_barcodes/mpra3_annot_THP1_IFNB_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
                  
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          )                         
res <-testLrt(obj)
write.csv(res,"20240616_comparative_THP1_IFNB_alleleOnly.csv")

results <- process_datasets(
    "read_counts_R1R2/allele_only/THP1_LPSIFNG_DNA_matched_barcodes_reshaped_allele.csv",
    "read_counts_R1R2/allele_only/THP1_LPSIFNG_RNA_matched_barcodes_reshaped_allele.csv",
    "annotation_barcodes/mpra3_annot_THP1_LPSIFNG_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )

obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          )                         
res <-testLrt(obj)
write.csv(res,"20240616_comparative_THP1_LPSIFNG_alleleOnly.csv")

##############################################################
results <- process_datasets(
    "read_counts_R1R2/allele_only/THP1_IFNG_DNA_matched_barcodes_reshaped_allele.csv",
    "read_counts_R1R2/allele_only/THP1_IFNG_RNA_matched_barcodes_reshaped_allele.csv",
    "annotation_barcodes/mpra3_annot_THP1_IFNG_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )                         
res <-testLrt(obj)
write.csv(res,"20240616_comparative_THP1_IFNG_alleleOnly.csv")



results <- process_datasets(
    "read_counts_R1R2/allele_only/THP1_IFNBvsNaive_DNA_matched_barcodes_reshaped_allele.csv",
    "read_counts_R1R2/allele_only/THP1_IFNBvsNaive_RNA_matched_barcodes_reshaped_allele.csv",
    "annotation_barcodes/mpra3_annot_THP1_IFNBvsNaive_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   


obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+IFNB_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )                 
res <-testLrt(obj)
write.csv(res,"20240812_comparative_THP1_IFNBvsNaive_interaction.csv")


results <- process_datasets(
    "read_counts_R1R2/allele_only/THP1_IFNGvsNaive_DNA_matched_barcodes_reshaped_allele.csv",
    "read_counts_R1R2/allele_only/THP1_IFNGvsNaive_RNA_matched_barcodes_reshaped_allele.csv",
    "annotation_barcodes/mpra3_annot_THP1_IFNGvsNaive_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   


obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+IFNG_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )               
res <-testLrt(obj)
write.csv(res,"20240812_comparative_THP1_IFNGvsNaive_interaction.csv")


df_DNA <- read.csv("read_counts_R1R2/allele_only/THP1_LPSIFNGvsIFNG_DNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/allele_only/THP1_LPSIFNGvsIFNG_RNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_THP1_LPSIFNGvsIFNG_barcodes.csv", header=TRUE,row.names=1)

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID

##############################################################
total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]
df_DNA <- df_DNA[, selected_columns]
df_RNA <- df_RNA[, selected_columns]
annot_DNA <- annot_DNA[selected_columns,]
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   


obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+LPSIFNG_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )                     
res <-testLrt(obj)
write.csv(res,"20240812_comparative_THP1_LPSIFNGvsIFNG_interaction.csv")

df_DNA <- read.csv("read_counts_R1R2/allele_only/THP1_LPSIFNGvsNaive_DNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/allele_only/THP1_LPSIFNGvsNaive_RNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_THP1_LPSIFNGvsNaive_barcodes.csv", header=TRUE,row.names=1)

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID

##############################################################
total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]
df_DNA <- df_DNA[, selected_columns]
df_RNA <- df_RNA[, selected_columns]
annot_DNA <- annot_DNA[selected_columns,]
##############################################################

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   


obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+LPSIFNG_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )              
res <-testLrt(obj)
write.csv(res,"20240812_comparative_THP1_LPSIFNGvsNaive_interaction.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-06-17 17:42:33.342637
Success: TRUE

Task duration:
    user   system  elapsed 
1019.238 3692.237 9238.075 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  5710399 305.0    9001995 480.8  9001995 480.8
Vcells 11410494  87.1   22045152 168.2 22044750 168.2

Log messages:
INFO [2024-06-17 15:08:36] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 16
Node: 9
Timestamp: 2024-06-17 18:20:53.83615
Success: TRUE

Task duration:
     user    system   elapsed 
 1285.305  4691.823 11541.216 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  5711485 305.1    9001995 480.8  9001995 480.8
Vcells 11415689  87.1   22045152 168.2 22044750 168.2

Log messages:
INFO [2024-06-17 15:08:33] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 23
Node: 2
Timestamp

In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 0) Input files
# =========================================================
files = {
    "THP1_IFNB": "allele_differences_withoutcontrol/20240813_allele_only/20240616_comparative_THP1_IFNB_alleleOnly.csv",
    "THP1_IFNG": "allele_differences_withoutcontrol/20240813_allele_only/20240616_comparative_THP1_IFNG_alleleOnly.csv",
    "THP1_LPSIFNG": "allele_differences_withoutcontrol/20240813_allele_only/20240616_comparative_THP1_LPSIFNG_alleleOnly.csv",
    "THP1_Naive": "allele_differences_withoutcontrol/20240813_allele_only/20240616_comparative_THP1_Naive_alleleOnly.csv",
}

# Output directory
out_dir = "allele_differences_withoutcontrol/20240813_allele_only/thp1_consistency_outputs"
os.makedirs(out_dir, exist_ok=True)

# =========================================================
# 1) Robust column detection
# =========================================================
def find_col(cols, candidates):
    lower_map = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    # fallback substring
    for c in cols:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

dfs = []
used_cols = []

for cond, path in files.items():
    d0 = pd.read_csv(path, index_col=0)
    logfc_col = find_col(d0.columns, ["logFC", "log2FoldChange", "effect", "coef"])
    pval_col = find_col(d0.columns, ["pval", "p.value", "P.Value", "p_value"])
    fdr_col = find_col(d0.columns, ["fdr", "padj", "adj.P.Val", "qval", "q_value"])
    if not all([logfc_col, pval_col, fdr_col]):
        raise ValueError(f"{cond}: could not detect required columns among {list(d0.columns)}")
    used_cols.append((cond, logfc_col, pval_col, fdr_col))
    d = d0[[logfc_col, pval_col, fdr_col]].copy()
    d.columns = [f"{cond}_logFC", f"{cond}_pval", f"{cond}_fdr"]
    dfs.append(d)

df = pd.concat(dfs, axis=1, join="inner").copy()

# thresholds
FDR_THRESH = 0.05
PVAL_THRESH = 0.05
conditions = list(files.keys())

# =========================================================
# 2) Pairwise effect-size correlations
# =========================================================
pearson_mat = pd.DataFrame(index=conditions, columns=conditions, dtype=float)
pairwise_stats = []

for c1, c2 in itertools.combinations(conditions, 2):
    tmp = df[[f"{c1}_logFC", f"{c2}_logFC"]].dropna()
    pearson_r = tmp.iloc[:, 0].corr(tmp.iloc[:, 1], method="pearson")
    spearman_r = tmp.iloc[:, 0].corr(tmp.iloc[:, 1], method="spearman")
    pearson_mat.loc[c1, c2] = pearson_r
    pearson_mat.loc[c2, c1] = pearson_r
    pairwise_stats.append({
        "cond1": c1,
        "cond2": c2,
        "n": len(tmp),
        "pearson_r": pearson_r,
        "spearman_r": spearman_r,
    })

for c in conditions:
    pearson_mat.loc[c, c] = 1.0

pairwise_stats_df = pd.DataFrame(pairwise_stats).sort_values("pearson_r", ascending=False)
pairwise_stats_df.to_csv(os.path.join(out_dir, "pairwise_logFC_correlations.csv"), index=False)

# =========================================================
# 3) Significance consistency summary
# =========================================================
def summarize_consistency(df, sig_suffix, sig_thresh, label):
    sig_mat = pd.DataFrame(index=df.index)
    sign_mat = pd.DataFrame(index=df.index)

    for c in conditions:
        sig_mat[c] = (df[f"{c}_{sig_suffix}"] < sig_thresh).astype(int)
        sign_mat[c] = np.sign(df[f"{c}_logFC"])

    n_sig = sig_mat.sum(axis=1)

    def direction_consistent(row_sig, row_sign):
        active = row_sig == 1
        vals = row_sign[active]
        vals = vals[vals != 0]
        if len(vals) <= 1:
            return np.nan
        return int(len(set(vals)) == 1)

    direction_cons = pd.Series(
        [direction_consistent(sig_mat.loc[idx], sign_mat.loc[idx]) for idx in df.index],
        index=df.index,
        name=f"{label}_direction_consistent",
    )

    out = pd.DataFrame(index=df.index)
    out[f"{label}_n_sig"] = n_sig
    out[f"{label}_direction_consistent"] = direction_cons

    summary_rows = []
    for k in [1, 2, 3, 4]:
        mask = n_sig >= k
        n_snps = int(mask.sum())
        if k == 1:
            n_dir_cons = np.nan
            frac_dir_cons = np.nan
        else:
            valid = direction_cons[mask].dropna()
            n_dir_cons = int((valid == 1).sum()) if len(valid) else 0
            frac_dir_cons = float((valid == 1).mean()) if len(valid) else np.nan

        summary_rows.append({
            "criterion": f">={k}_conditions_significant",
            "n_snps": n_snps,
            "n_direction_consistent": n_dir_cons,
            "fraction_direction_consistent": frac_dir_cons,
        })

    return out, sig_mat, pd.DataFrame(summary_rows)

fdr_out, fdr_sig_mat, fdr_summary = summarize_consistency(df, "fdr", FDR_THRESH, "FDR")
pval_out, pval_sig_mat, pval_summary = summarize_consistency(df, "pval", PVAL_THRESH, "PVAL")

fdr_summary.to_csv(os.path.join(out_dir, "consistency_summary_FDR.csv"), index=False)
pval_summary.to_csv(os.path.join(out_dir, "consistency_summary_PVAL.csv"), index=False)

# per-condition counts
per_condition_counts = []
for c in conditions:
    per_condition_counts.append({
        "condition": c,
        "n_fdr_sig": int((df[f"{c}_fdr"] < FDR_THRESH).sum()),
        "n_pval_sig": int((df[f"{c}_pval"] < PVAL_THRESH).sum()),
    })
per_condition_counts_df = pd.DataFrame(per_condition_counts)
per_condition_counts_df.to_csv(os.path.join(out_dir, "per_condition_significant_counts.csv"), index=False)

# Save merged table
df_out = pd.concat([df, fdr_out, pval_out], axis=1)
df_out.to_csv(os.path.join(out_dir, "merged_THP1_consistency_table.csv"))

# =========================================================
# 4) Plot 1: Pearson correlation heatmap
# =========================================================
sns.set(style="ticks", context="paper")
plt.figure(figsize=(5.8, 5.0))
sns.heatmap(
    pearson_mat.astype(float),
    annot=True,
    fmt=".2f",
    cmap="Blues",
    vmin=0,
    vmax=1,
    square=True,
    cbar_kws={"label": "Pearson r"},
    linewidths=0.5
)
plt.title("THP-1 cross-condition allelic effect correlation")
plt.tight_layout()
heatmap_pdf = os.path.join(out_dir, "THP1_logFC_correlation_heatmap.pdf")
heatmap_png = os.path.join(out_dir, "THP1_logFC_correlation_heatmap.png")
plt.savefig(heatmap_pdf, bbox_inches="tight")
plt.savefig(heatmap_png, dpi=600, bbox_inches="tight")
plt.close()

# =========================================================
# 5) Plot 2: how many conditions significant
# =========================================================
fdr_nsig_counts = fdr_out["FDR_n_sig"].value_counts().reindex([0, 1, 2, 3, 4], fill_value=0)
pval_nsig_counts = pval_out["PVAL_n_sig"].value_counts().reindex([0, 1, 2, 3, 4], fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
axes[0].bar(fdr_nsig_counts.index.astype(str), fdr_nsig_counts.values)
axes[0].set_title("FDR < 0.05")
axes[0].set_xlabel("Number of THP-1 conditions significant")
axes[0].set_ylabel("Number of SNPs")

axes[1].bar(pval_nsig_counts.index.astype(str), pval_nsig_counts.values)
axes[1].set_title("P < 0.05")
axes[1].set_xlabel("Number of THP-1 conditions significant")

for ax in axes:
    ax.tick_params(axis="both", direction="in", top=True, right=True)

plt.tight_layout()
bar_pdf = os.path.join(out_dir, "THP1_significant_condition_counts.pdf")
bar_png = os.path.join(out_dir, "THP1_significant_condition_counts.png")
plt.savefig(bar_pdf, bbox_inches="tight")
plt.savefig(bar_png, dpi=600, bbox_inches="tight")
plt.close()

# =========================================================
# 6) Plot 3: Pairwise scatter matrix
# =========================================================
pairs = list(itertools.combinations(conditions, 2))
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
axes = axes.flatten()

for ax, (c1, c2) in zip(axes, pairs):
    tmp = df[[f"{c1}_logFC", f"{c2}_logFC"]].dropna()
    r = tmp.iloc[:, 0].corr(tmp.iloc[:, 1], method="pearson")
    ax.scatter(tmp.iloc[:, 0], tmp.iloc[:, 1], s=8, alpha=0.35)
    lim = np.nanmax(np.abs(tmp.values))
    lim = max(1, float(lim))
    ax.plot([-lim, lim], [-lim, lim], linewidth=1)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_title(f"{c1} vs {c2}\nr = {r:.2f}", fontsize=10)
    ax.set_xlabel(c1)
    ax.set_ylabel(c2)
    ax.tick_params(direction="in", top=True, right=True)

plt.tight_layout()
scatter_pdf = os.path.join(out_dir, "THP1_pairwise_logFC_scatterplots.pdf")
scatter_png = os.path.join(out_dir, "THP1_pairwise_logFC_scatterplots.png")
plt.savefig(scatter_pdf, bbox_inches="tight")
plt.savefig(scatter_png, dpi=600, bbox_inches="tight")
plt.close()

# =========================================================
# 7) Display summary table
# =========================================================
summary = {
    "Used columns": pd.DataFrame(used_cols, columns=["condition", "logFC_col", "pval_col", "fdr_col"]),
    "Pairwise correlations": pairwise_stats_df,
    "Per-condition counts": per_condition_counts_df,
    "FDR consistency": fdr_summary,
    "PVAL consistency": pval_summary,
}
for name, df_show in summary.items():
    print(f"\n=== {name} ===")
    print(df_show.to_string(index=False))
print("\nSaved files:")
for p in [heatmap_pdf, heatmap_png, bar_pdf, bar_png, scatter_pdf, scatter_png]:
    print(p)


Saved:
allele_differences_withoutcontrol/20240813_allele_only/thp1_consistency_outputs\THP1_significant_condition_counts_annotated.pdf
allele_differences_withoutcontrol/20240813_allele_only/thp1_consistency_outputs\THP1_significant_condition_counts_annotated.png
allele_differences_withoutcontrol/20240813_allele_only/thp1_consistency_outputs\THP1_consistency_summary_richer.pdf
allele_differences_withoutcontrol/20240813_allele_only/thp1_consistency_outputs\THP1_consistency_summary_richer.png


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

out_dir = "allele_differences_withoutcontrol/20240813_allele_only/thp1_consistency_outputs"

# Load previously saved summaries
fdr_summary = pd.read_csv(os.path.join(out_dir, "consistency_summary_FDR.csv"))
pval_summary = pd.read_csv(os.path.join(out_dir, "consistency_summary_PVAL.csv"))
per_condition_counts_df = pd.read_csv(os.path.join(out_dir, "per_condition_significant_counts.csv"))
merged = pd.read_csv(os.path.join(out_dir, "merged_THP1_consistency_table.csv"), index_col=0)

# Recompute exact counts of SNPs significant in exactly k conditions
fdr_exact = merged["FDR_n_sig"].value_counts().reindex([0,1,2,3,4], fill_value=0).sort_index()
pval_exact = merged["PVAL_n_sig"].value_counts().reindex([0,1,2,3,4], fill_value=0).sort_index()

n_total = len(merged)

# Direction-consistency among SNPs significant in exactly k conditions
def exact_direction_table(nsig_col, dir_col):
    rows = []
    for k in [2,3,4]:
        mask = merged[nsig_col] == k
        sub = merged.loc[mask, dir_col].dropna()
        n = int(mask.sum())
        n_cons = int((sub == 1).sum()) if len(sub) else 0
        frac = (n_cons / n) if n > 0 else np.nan
        rows.append({"k": k, "n": n, "n_consistent": n_cons, "fraction_consistent": frac})
    return pd.DataFrame(rows)

fdr_exact_dir = exact_direction_table("FDR_n_sig", "FDR_direction_consistent")
pval_exact_dir = exact_direction_table("PVAL_n_sig", "PVAL_direction_consistent")

# Plot
sns.set(style="ticks", context="paper")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharey=True)

plot_specs = [
    (axes[0], fdr_exact, fdr_exact_dir, "FDR < 0.05"),
    (axes[1], pval_exact, pval_exact_dir, "P < 0.05"),
]

for ax, counts, dir_df, title in plot_specs:
    x = np.arange(len(counts))
    vals = counts.values
    
    bars = ax.bar(x, vals)
    ax.set_xticks(x)
    ax.set_xticklabels([str(i) for i in counts.index])
    ax.set_xlabel("Number of THP-1 conditions significant")
    ax.set_title(title)
    ax.tick_params(axis="both", direction="in", top=True, right=True)
    
    # numeric count + percentage on each bar
    ymax = max(vals) * 1.28 if max(vals) > 0 else 1
    ax.set_ylim(0, ymax)
    
    for xi, v in zip(x, vals):
        pct = 100 * v / n_total
        ax.text(
            xi, v + ymax*0.02,
            f"{int(v)}\n({pct:.1f}%)",
            ha="center", va="bottom", fontsize=9
        )
    
    # direction consistency annotation for k>=2
    for _, row in dir_df.iterrows():
        k = int(row["k"])
        frac = row["fraction_consistent"]
        n_cons = int(row["n_consistent"])
        n = int(row["n"])
        xi = list(counts.index).index(k)
        if n > 0:
            ax.text(
                xi, ymax*0.83,
                f"dir. consistent:\n{n_cons}/{n} ({100*frac:.1f}%)",
                ha="center", va="top", fontsize=8
            )

axes[0].set_ylabel("Number of SNPs")

# Add a concise figure note across bottom
fig.text(
    0.5, -0.02,
    f"Total SNPs tested = {n_total}. Bars show SNPs significant in exactly 0, 1, 2, 3, or 4 THP-1 conditions.",
    ha="center", va="top", fontsize=10
)

plt.tight_layout()

out_pdf = os.path.join(out_dir, "THP1_significant_condition_counts_annotated.pdf")
out_png = os.path.join(out_dir, "THP1_significant_condition_counts_annotated.png")
plt.savefig(out_pdf, bbox_inches="tight")
plt.savefig(out_png, dpi=600, bbox_inches="tight")
plt.close()

# Also create a more information-dense supplementary panel:
# left = per-condition sig counts; right = exact significant-count bars with labels
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))

# Panel A: per-condition counts
x = np.arange(len(per_condition_counts_df))
w = 0.38
axes[0].bar(x - w/2, per_condition_counts_df["n_fdr_sig"], width=w, label="FDR < 0.05")
axes[0].bar(x + w/2, per_condition_counts_df["n_pval_sig"], width=w, label="P < 0.05")
axes[0].set_xticks(x)
axes[0].set_xticklabels(per_condition_counts_df["condition"], rotation=30, ha="right")
axes[0].set_ylabel("Number of significant SNPs")
axes[0].set_title("Per-condition significant SNPs")
axes[0].legend(frameon=True, fontsize=9)
axes[0].tick_params(axis="both", direction="in", top=True, right=True)
for container in axes[0].containers:
    for bar in container:
        h = bar.get_height()
        axes[0].text(bar.get_x()+bar.get_width()/2, h+3, f"{int(h)}", ha="center", va="bottom", fontsize=8)

# Panel B: exact FDR counts
counts = fdr_exact
x = np.arange(len(counts))
bars = axes[1].bar(x, counts.values)
axes[1].set_xticks(x)
axes[1].set_xticklabels([str(i) for i in counts.index])
axes[1].set_title("Exact number of THP-1 conditions\nwith FDR-significant effects")
axes[1].set_xlabel("Number of significant conditions")
axes[1].tick_params(axis="both", direction="in", top=True, right=True)
ymax = max(counts.values) * 1.2
axes[1].set_ylim(0, ymax)
for xi, v in zip(x, counts.values):
    axes[1].text(xi, v + ymax*0.02, f"{int(v)}", ha="center", va="bottom", fontsize=9)

# Panel C: exact nominal counts
counts = pval_exact
x = np.arange(len(counts))
bars = axes[2].bar(x, counts.values)
axes[2].set_xticks(x)
axes[2].set_xticklabels([str(i) for i in counts.index])
axes[2].set_title("Exact number of THP-1 conditions\nwith nominally significant effects")
axes[2].set_xlabel("Number of significant conditions")
axes[2].tick_params(axis="both", direction="in", top=True, right=True)
ymax = max(counts.values) * 1.2
axes[2].set_ylim(0, ymax)
for xi, v in zip(x, counts.values):
    axes[2].text(xi, v + ymax*0.02, f"{int(v)}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
rich_pdf = os.path.join(out_dir, "THP1_consistency_summary_richer.pdf")
rich_png = os.path.join(out_dir, "THP1_consistency_summary_richer.png")
plt.savefig(rich_pdf, bbox_inches="tight")
plt.savefig(rich_png, dpi=600, bbox_inches="tight")
plt.close()

print("Saved:")
print(out_pdf)
print(out_png)
print(rich_pdf)
print(rich_png)


Saved:
allele_differences_withoutcontrol/20240813_allele_only/thp1_consistency_outputs\THP1_significant_condition_counts_annotated.pdf
allele_differences_withoutcontrol/20240813_allele_only/thp1_consistency_outputs\THP1_significant_condition_counts_annotated.png
allele_differences_withoutcontrol/20240813_allele_only/thp1_consistency_outputs\THP1_consistency_summary_richer.pdf
allele_differences_withoutcontrol/20240813_allele_only/thp1_consistency_outputs\THP1_consistency_summary_richer.png


: 